# Box Office Bomb Data Pipeline Assignment

**Total Marks: 20**

This notebook implements an end-to-end pipeline to:
1. Scrape box office bomb data from Wikipedia
2. Validate & clean data using Pydantic
3. Enrich with OMDb API data
4. Perform consistency checks
5. Create final categorized dataset

## Setup and Imports

In [ ]:
# Import required libraries
import pandas as pd
import requests
from bs4 import BeautifulSoup
from pydantic import BaseModel, field_validator, ValidationError
from typing import Optional
import re
import time
from pathlib import Path

## Task 1: Scrape the "Bombs" Table (4 Marks)

Extract raw data from the Wikipedia HTML file:
- Film Title (with symbols like § and †)
- Year
- Net production budget (may contain ranges like "$100–160")
- Estimated loss (Nominal column)

Note:
- You need to extract the entire raw string with the symbols, references, etc along with the titles.
- You must handle the nested headers in the Wikipedia table (Budget and Loss columns have sub-headers).

In [ ]:
def scrape_box_office_bombs():
    """
    Scrape the box office bombs table from the local HTML file.
    Returns a list of dictionaries with raw extracted data.
    """
    # TODO: Load the HTML file
    url="https://en.wikipedia.org/wiki/List_of_biggest_box-office_bombs"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36" #->direct response is not given wikipedia source,so we act like we are a chrome browser to fetch results from it
    }
    response=requests.get(url,headers=headers)

    soup=BeautifulSoup(response.text,"html.parser")

    #print(soup)

    # TODO: Parse with BeautifulSoup


    # TODO: Find the main table - it's wikitable sortable with caption "Biggest box-office bombs"
    details=soup.find("table",{"class":"wikitable sortable plainrowheaders"})

    #print(details)
    details1=details.find_all("tr")
    #print(details1)
    details2=details1[2:]
    raw_data = []
    for i in details2:
      cells=i.find_all(["th","td"])
      movie_entry={'raw_title':cells[0].get_text(strip=True),"raw_year":cells[1].get_text(strip=True),"raw_budget":cells[2].get_text(strip=True),"raw_loss":cells[4].get_text(strip=True)}
      raw_data.append(movie_entry)

    # TODO: Iterate through rows (skip headers) and extract: 'raw_title', 'raw_year', 'raw_budget', 'raw_loss'.
    # Store extracted data in raw_data. Example entry: {'raw_title': 'Town & Country', 'raw_year': '2001', 'raw_budget': '$90', 'raw_loss': '$10.4'}

    return raw_data

raw=scrape_box_office_bombs()
print(raw)

[{'raw_title': 'The 13th Warrior', 'raw_year': '1999', 'raw_budget': '$100–160', 'raw_loss': '$69–129'}, {'raw_title': 'The 355', 'raw_year': '2022', 'raw_budget': '$40–75', 'raw_loss': '$93'}, {'raw_title': '47 Ronin', 'raw_year': '2013', 'raw_budget': '$175–225', 'raw_loss': '$96'}, {'raw_title': 'The Adventures of Baron Munchausen', 'raw_year': '1988', 'raw_budget': '$46.6', 'raw_loss': '$38.5'}, {'raw_title': 'The Adventures of Pluto Nash', 'raw_year': '2002', 'raw_budget': '$100', 'raw_loss': '$96'}, {'raw_title': 'The Adventures of Rocky & Bullwinkle', 'raw_year': '2000', 'raw_budget': '$76–98.6', 'raw_loss': '$63.5'}, {'raw_title': 'The Alamo', 'raw_year': '2004', 'raw_budget': '$107', 'raw_loss': '$94'}, {'raw_title': 'Alexander', 'raw_year': '2004', 'raw_budget': '$155', 'raw_loss': '$71'}, {'raw_title': 'Ali', 'raw_year': '2001', 'raw_budget': '$107', 'raw_loss': '$63'}, {'raw_title': 'Allied', 'raw_year': '2016', 'raw_budget': '$85', 'raw_loss': '$75–90'}, {'raw_title': 'Ams

In [ ]:
# Test the scraping function
raw_movies = scrape_box_office_bombs()
print(f"Scraped {len(raw_movies)} movies")
print("\nLast 15 raw entries:")
for i, movie in enumerate(raw_movies[-15:]):
    print(f"\n{i+1}. {movie}")

Scraped 139 movies

Last 15 raw entries:

1. {'raw_title': 'Town & Country', 'raw_year': '2001', 'raw_budget': '$90', 'raw_loss': '$85'}

2. {'raw_title': 'Transformers: The Last Knight', 'raw_year': '2017', 'raw_budget': '$217–260', 'raw_loss': '$100+'}

3. {'raw_title': 'Treasure Planet', 'raw_year': '2002', 'raw_budget': '$140', 'raw_loss': '$85'}

4. {'raw_title': 'Tron: Ares†', 'raw_year': '2025', 'raw_budget': '$220', 'raw_loss': '$132.7'}

5. {'raw_title': 'Turning Red§', 'raw_year': '2022', 'raw_budget': '$175', 'raw_loss': '$173'}

6. {'raw_title': 'Valerian and the City of a Thousand Planets', 'raw_year': '2017', 'raw_budget': '$177.2–180', 'raw_loss': '$82'}

7. {'raw_title': 'West Side Story', 'raw_year': '2021', 'raw_budget': '$100', 'raw_loss': '$104'}

8. {'raw_title': 'Wild Wild West', 'raw_year': '1999', 'raw_budget': '$170', 'raw_loss': '$66.2'}

9. {'raw_title': 'Windtalkers', 'raw_year': '2002', 'raw_budget': '$115–120', 'raw_loss': '$76–81'}

10. {'raw_title': 'Wis

## Task 2: Pydantic Data Parsing & Validation (6 Marks)

Create a Pydantic model that:
- Cleans titles (removes §, †, and footnotes like [nb 2], [1])
- Parses numeric values (handles ranges, currency symbols)
- Validates year as integer

In [ ]:
class MovieData(BaseModel):
    """
    Pydantic model for validating and cleaning movie data.
    """
    # TODO: Define the 4 required fields with their types:
    # title (str), year (int), budget_millions (float), loss_millions (float)
    title:str
    year:int
    budget_millions:float
    loss_millions:float
    @field_validator('title', mode='before')
    @classmethod
    def clean_title(cls, v):
        """
        Remove footnote markers and special symbols from title.
        - Remove § (streaming symbol)
        - Remove † (currently playing symbol)
        - Remove footnotes like [nb 2], [1], etc.
        """
        # TODO: Implement title cleaning logic
        # Hint: Use .replace() for symbols and re.sub() for footnotes

        # v=re.sub(r'[^a-zA-Z0-9\s]','',v)

        v = v.replace('§', '')
        v = v.replace('†', '')

        # Remove footnotes like [nb 2], [1], [nb 5], etc.
        v = re.sub(r'\[.*?\]', '', v)

        # Strip extra whitespace
        v = v.strip()


        return v

    @field_validator('year', mode='before')
    @classmethod
    def validate_year(cls, v):
        """
        Ensure year is a valid integer.
        Remove any extra characters and convert to int.
        """
        # TODO: Clean string (remove non-digits) and convert to integer
        v = re.sub(r"\D", "", str(v))   # keeping only digits
        return int(v)


    @field_validator('budget_millions', mode='before')
    @classmethod
    def parse_budget(cls, v):
        """
        Parse budget value:
        - Strip $ and other currency symbols
        - Handle ranges (e.g., "100–160") by calculating average
        - Remove reference tags
        """
        # TODO: Call your helper method _parse_numeric_value

        v = re.sub(r"\[.*?\]", "", str(v))
        v=v.replace("$",'')
        v=v.replace("'","")
        if '\u2013' in v:
          v=v.split('\u2013')
          v=(float(v[0])+float(v[1]))/2
          return v
        else:
          return float(v)


    @field_validator('loss_millions', mode='before')
    @classmethod
    def parse_loss(cls, v):
        """
        Parse loss value with same logic as budget.
        """
        # TODO: Call your helper method _parse_numeric_value

        v=v.replace("'","")
        v1=cls._parse_numeric_value(v)
        return v1

    @staticmethod
    def _parse_numeric_value(v):
        """
        Helper method to parse numeric values with ranges.
        """
        # TODO: Implement logic to:
        # 1. Clean string (remove $, reference tags)
        # 2. Check for ranges (hyphen or en-dash) and calculate average
        # 3. Return a float
        v = re.sub(r"\[.*?\]", "", str(v))
        v=v.replace("$",'')
        v=v.replace("+",'')              #by checking data in this column only one movies has budget $100+
        if '\u2013' in v:
          v=v.split('\u2013')
          v=(float(v[0])+float(v[1]))/2
          return v
        else:
          return float(v)


In [ ]:
# Validate and clean the raw data
validated_movies = []
failed_validations = []

for raw_movie in raw_movies:
    try:
        movie = MovieData(
            title=raw_movie['raw_title'],
            year=raw_movie['raw_year'],
            budget_millions=raw_movie['raw_budget'],
            loss_millions=raw_movie['raw_loss']
        )
        validated_movies.append(movie)
    except ValidationError as e:
        failed_validations.append({
            'raw_data': raw_movie,
            'error': str(e)
        })
        print(f"Validation failed for {raw_movie['raw_title']}: {e}")

print(f"\n{'='*60}")
print(f"Validation Results:")
print(f"Successfully validated: {len(validated_movies)} movies")
print(f"Failed validations: {len(failed_validations)} movies")
print(f"{'='*60}")

# Show first 3 validated movies
print("\nLast 15 validated movies:")
for i, movie in enumerate(validated_movies[-15:]):
    print(f"\n{i+1}. {movie.model_dump()}")


Validation Results:
Successfully validated: 139 movies
Failed validations: 0 movies

Last 15 validated movies:

1. {'title': 'Town & Country', 'year': 2001, 'budget_millions': 90.0, 'loss_millions': 85.0}

2. {'title': 'Transformers: The Last Knight', 'year': 2017, 'budget_millions': 238.5, 'loss_millions': 100.0}

3. {'title': 'Treasure Planet', 'year': 2002, 'budget_millions': 140.0, 'loss_millions': 85.0}

4. {'title': 'Tron: Ares', 'year': 2025, 'budget_millions': 220.0, 'loss_millions': 132.7}

5. {'title': 'Turning Red', 'year': 2022, 'budget_millions': 175.0, 'loss_millions': 173.0}

6. {'title': 'Valerian and the City of a Thousand Planets', 'year': 2017, 'budget_millions': 178.6, 'loss_millions': 82.0}

7. {'title': 'West Side Story', 'year': 2021, 'budget_millions': 100.0, 'loss_millions': 104.0}

8. {'title': 'Wild Wild West', 'year': 1999, 'budget_millions': 170.0, 'loss_millions': 66.2}

9. {'title': 'Windtalkers', 'year': 2002, 'budget_millions': 117.5, 'loss_millions': 

## Task 3: Enrich with OMDb Data (4 Marks)

Query OMDb API for each movie to get:
- Plot
- Metascore
- IMDb Rating
- Director
- Language

Handle API failures (Response='False') or missing fields ('N/A') gracefully by storing them as None. Do not delete the row.

In [ ]:
# OMDb API configuration
# NOTE: You need to get a free API key from http://www.omdbapi.com/apikey.aspx
OMDB_API_KEY = "d88e5bee"  # Replace with your actual API key
OMDB_BASE_URL = "http://www.omdbapi.com/"

def query_omdb(title: str, year: int) -> dict:
    OMDB_API_KEY = "d88e5bee"
    """
    Query OMDb API for movie metadata.
    Returns dict with plot, metascore, imdb_rating, director, language, omdb_year.
    Returns None values if movie not found or API fails.
    """
    # TODO: Construct params and make GET request
    params={'apikey':OMDB_API_KEY,'t':title,'y':year}
    response=requests.get(OMDB_BASE_URL,params=params)
    data=response.json()
    send={}

    # TODO: Handle 'Response': 'False'
    if data['Response']=='False':
      send['plot']=  None
      send['metascore']= None
      send['imdb_rating']= None
      send['director']= None
      send['language']= None
      send['omdb_year']=None
    else:
      send['plot']= data['Plot'] if data['Plot'] !='N/A' else None
      send['metascore']=data['Metascore'] if data['Metascore'] !='N/A' else None
      send['imdb_rating']=data['imdbRating']  if data['imdbRating'] !='N/A' else None
      send['director']=data['Director'] if data['Director'] !='N/A' else None
      send['language']=data['Language'] if data['Language'] !='N/A' else None
      send['omdb_year']=data['Year'] if data['Year'] !='N/A' else None


    # TODO: Extract fields and handle 'N/A' conversion to None/Numbers

    return send

In [ ]:
# Enrich each validated movie with OMDb data
enriched_data = []

print("Querying OMDb API...")
# TODO: Loop through validated_movies, call query_omdb, and merge data
for i in validated_movies:
    # Query OMDb
    omdb_data = query_omdb(i.title, i.year)

    # Merge Wikipedia (validated) data + OMDb data
    enriched_movie = {
        "title": i.title,
        "wiki_year": i.year,
        "budget_millions": i.budget_millions,
        "loss_millions": i.loss_millions,
        **omdb_data
    }

    enriched_data.append(enriched_movie)
    time.sleep(0.1)

print(f"\nEnriched {len(enriched_data)} movies with OMDb data")

# Show sample enriched data
print("\nFirst enriched entry:")
print(enriched_data[0])

Querying OMDb API...

Enriched 139 movies with OMDb data

First enriched entry:
{'title': 'The 13th Warrior', 'wiki_year': 1999, 'budget_millions': 130.0, 'loss_millions': 99.0, 'plot': 'A man, having fallen in love with the wrong woman, is sent by the sultan himself on a diplomatic mission to a distant land as an ambassador. Stopping at a Viking village port to restock on supplies, he finds himself unwittingly em...', 'metascore': '42', 'imdb_rating': '6.6', 'director': 'John McTiernan', 'language': 'English, Latin, Swedish, Norse, Old, Danish, Arabic', 'omdb_year': '1999'}


## Task 4: Data Consistency Check (2 Marks)

Compare Wikipedia year with OMDb year:
- "Verified": Years match (±1 year tolerance)
- "Mismatch": Years differ by >1
- "Not Found": OMDb returned no data

In [ ]:
def determine_match_status(wiki_year: int, omdb_year: Optional[int]) -> str:
    """
    Determine the match status between Wikipedia and OMDb years.
    Returns "Verified", "Mismatch", or "Not Found".
    """
    # TODO: Implement logic
    if omdb_year is None:
        return "Not Found"

    try:
        wiki_year = int(wiki_year)
        omdb_year = int(omdb_year)
    except:
        return "Not Found"

    if abs(wiki_year - omdb_year) <= 1:
        return "Verified"
    else:
        return "Mismatch"

# TODO: Apply this function to your enriched_data list and add match_status to each entry
for movie in enriched_data:
    movie["match_status"] = determine_match_status(
        movie["wiki_year"],
        movie.get("omdb_year")
    )

# Show match status distribution
df_temp = pd.DataFrame(enriched_data)
print("Match Status Distribution:")
print(df_temp['match_status'].value_counts())

# Show some mismatches if any
mismatches = df_temp[df_temp['match_status'] == 'Mismatch']
if len(mismatches) > 0:
    print(f"\nSample Mismatches (showing up to 5):")
    print(mismatches[['title', 'wiki_year', 'omdb_year', 'match_status']].head())

Match Status Distribution:
match_status
Verified     137
Not Found      2
Name: count, dtype: int64


In [ ]:
# Filter rows where match_status is 'Not Found'
not_found = df_temp[df_temp['match_status'] == 'Not Found']

if not not_found.empty:
    print("Movies with Year Not Found in OMDb (showing all):")
    print(not_found[['title', 'wiki_year', 'omdb_year', 'match_status']].to_string(index=False))

Movies with Year Not Found in OMDb (showing all):
                       title  wiki_year omdb_year match_status
        The Nutcracker in 3D       2010      None    Not Found
Snake Eyes: G.I. Joe Origins       2021      None    Not Found


## Task 5: Final Dataset & Categorization (4 Marks)

Create final DataFrame with:
- Loss_Category based on estimated loss:
  - "Catastrophic": Loss ≥ \$100M

  - "Severe": Loss between \$50M and \$100M

  - "Moderate": Loss < \$50M
- Save to `box_office_failures.csv`

Required columns: Title, Year, Director, Language, Budget_Millions, Loss_Millions, Loss_Category, IMDb_Rating, Metascore, Match_Status

In [ ]:
def categorize_loss(loss_millions: float) -> str:
    """
    Categorize the financial loss into tiers.
    """
    if loss_millions >= 100:
        return "Catastrophic"
    elif loss_millions >= 50:
        return "Severe"
    else:
        return "Moderate"

# Create DataFrame from enriched_data
df_final = pd.DataFrame(enriched_data)

# Add Loss_Category column
df_final['Loss_Category'] = df_final['loss_millions'].apply(categorize_loss)

# Select and rename columns to match requirements
# Required: Title, Year, Director, Language, Budget_Millions, Loss_Millions, Loss_Category, IMDb_Rating, Metascore, Match_Status
df_final = df_final.rename(columns={
    'title': 'Title',
    'wiki_year': 'Year',
    'director': 'Director',
    'language': 'Language',
    'budget_millions': 'Budget_Millions',
    'loss_millions': 'Loss_Millions',
    'imdb_rating': 'IMDb_Rating',
    'metascore': 'Metascore',
    'match_status': 'Match_Status'
})

# Select only required columns in specified order
df_final = df_final[['Title', 'Year', 'Director', 'Language', 'Budget_Millions',
                      'Loss_Millions', 'Loss_Category', 'IMDb_Rating', 'Metascore', 'Match_Status']]

# Display summary statistics
print("Final Dataset Summary:")
print(f"Total movies: {len(df_final)}")
print(f"\nLoss Category Distribution:")
print(df_final['Loss_Category'].value_counts())
print(f"\nBasic Statistics:")
print(df_final[['Budget_Millions', 'Loss_Millions', 'IMDb_Rating', 'Metascore']].describe())

# Display first few rows
print(f"\nFirst 10 rows of final dataset:")
df_final.head(10)


Final Dataset Summary:
Total movies: 139

Loss Category Distribution:
Loss_Category
Severe          85
Catastrophic    46
Moderate         8
Name: count, dtype: int64

Basic Statistics:
       Budget_Millions  Loss_Millions
count       139.000000     139.000000
mean        129.552518      91.917266
std          58.126686      32.407538
min          17.000000      10.800000
25%          81.250000      70.550000
50%         117.500000      85.000000
75%         175.000000     108.500000
max         326.000000     218.000000

First 10 rows of final dataset:


,Title,Year,Director,Language,Budget_Millions,Loss_Millions,Loss_Category,IMDb_Rating,Metascore,Match_Status
0,The 13th Warrior,1999,John McTiernan,"English, Latin, Swedish, Norse, Old, Danish, A...",130.0,99.0,Severe,6.6,42,Verified
1,The 355,2022,Simon Kinberg,"English, Chinese, Spanish, French, German, Ara...",57.5,93.0,Severe,5.6,40,Verified
2,47 Ronin,2013,Carl Rinsch,"English, Japanese",200.0,96.0,Severe,6.2,28,Verified
3,The Adventures of Baron Munchausen,1988,Terry Gilliam,English,46.6,38.5,Moderate,7.1,69,Verified
4,The Adventures of Pluto Nash,2002,Ron Underwood,English,100.0,96.0,Severe,3.9,12,Verified
5,The Adventures of Rocky & Bullwinkle,2000,Des McAnuff,English,87.3,63.5,Severe,4.3,36,Verified
6,The Alamo,2004,John Lee Hancock,"English, Spanish",107.0,94.0,Severe,6.1,47,Verified
7,Alexander,2004,Oliver Stone,English,155.0,71.0,Severe,5.6,40,Verified
8,Ali,2001,Michael Mann,"English, French, Swahili",107.0,63.0,Severe,6.7,65,Verified
9,Allied,2016,Robert Zemeckis,"English, French, German, Arabic",85.0,82.5,Severe,7.1,60,Verified


In [ ]:
# Save to CSV
output_path = 'box_office_failures.csv'

# Save DataFrame to CSV
df_final.to_csv(output_path, index=False)

print(f"✓ Dataset saved to: {output_path}")
print(f"✓ Total records: {len(df_final)}")


✓ Dataset saved to: box_office_failures.csv
✓ Total records: 139


In [ ]:
# Convert IMDb_Rating and Metascore to numeric types for analysis
df_final['IMDb_Rating'] = pd.to_numeric(df_final['IMDb_Rating'], errors='coerce')
df_final['Metascore'] = pd.to_numeric(df_final['Metascore'], errors='coerce')

## Additional Analysis (Optional)

In [ ]:
# Show some interesting insights
print("Top 10 Biggest Box Office Bombs (by loss):")
print(df_final.nlargest(10, 'Loss_Millions')[['Title', 'Year', 'Director', 'Loss_Millions', 'Loss_Category']])

print("\n" + "="*60)
print("\nMovies with Lowest IMDb Ratings:")
lowest_rated = df_final[df_final['IMDb_Rating'].notna()].nsmallest(5, 'IMDb_Rating')
print(lowest_rated[['Title', 'Year', 'Director', 'IMDb_Rating', 'Metascore', 'Loss_Millions']])

print("\n" + "="*60)
print("\nAverage Loss by Category:")
print(df_final.groupby('Loss_Category')['Loss_Millions'].agg(['mean', 'count']))

print("\n" + "="*60)
print("\nTop 5 Most Common Directors in Box Office Bombs:")
print(df_final['Director'].value_counts().head())

Top 10 Biggest Box Office Bombs (by loss):
                   Title  Year              Director  Loss_Millions  \
80           The Marvels  2023           Nia DaCosta          218.0   
116        Strange World  2022  Don Hall, Qui Nguyen          187.5   
77       The Lone Ranger  2013        Gore Verbinski          175.0   
86        Mortal Engines  2018      Christian Rivers          174.8   
128          Turning Red  2022             Domee Shi          173.0   
67           John Carter  2012        Andrew Stanton          156.0   
42             The Flash  2023       Andy Muschietti          155.0   
69         Jungle Cruise  2021    Jaume Collet-Serra          150.0   
68   Joker: Folie à Deux  2024         Todd Phillips          144.3   
87                 Mulan  2020             Niki Caro          141.0   

    Loss_Category  
80   Catastrophic  
116  Catastrophic  
77   Catastrophic  
86   Catastrophic  
128  Catastrophic  
67   Catastrophic  
42   Catastrophic  
69   Catastroph